In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# 1. Configuration
DATA_DIR = 'cell_images/' # Update this if your path is different
BATCH_SIZE = 32
IMG_SIZE = (150, 150)

# 2. Load and Split Dataset (80% Train, 20% Validation)
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

# Optimize performance for fast training
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

# 3. Build the Model (Transfer Learning)
base_model = EfficientNetB0(input_shape=(150, 150, 3), include_top=False, weights='imagenet')
base_model.trainable = False # Freeze base model layers initially

# Add custom classification head
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x) # Prevents overfitting
predictions = Dense(1, activation='sigmoid')(x) # Binary classification

model = Model(inputs=base_model.input, outputs=predictions)

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss='binary_crossentropy',
              metrics=['accuracy'])

# 4. Callbacks (Save the absolute best model)
callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    ModelCheckpoint('best_malaria_model.keras', monitor='val_accuracy', save_best_only=True)
]

# 5. Train the Model
print("Starting training...")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks
)
print("Training complete. Best model saved as 'best_malaria_model.keras'.")

Found 27558 files belonging to 2 classes.
Using 22047 files for training.
Found 27558 files belonging to 2 classes.
Using 5511 files for validation.
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 11s 1us/step
Starting training...
Epoch 1/10
689/689 ━━━━━━━━━━━━━━━━━━━━ 661s 912ms/step - accuracy: 0.8950 - loss: 0.2665 - val_accuracy: 0.9269 - val_loss: 0.1959
Epoch 2/10
689/689 ━━━━━━━━━━━━━━━━━━━━ 344s 500ms/step - accuracy: 0.9238 - loss: 0.2013 - val_accuracy: 0.9272 - val_loss: 0.1882
Epoch 3/10
689/689 ━━━━━━━━━━━━━━━━━━━━ 301s 437ms/step - accuracy: 0.9269 - loss: 0.1929 - val_accuracy: 0.9340 - val_loss: 0.1732
Epoch 4/10
689/689 ━━━━━━━━━━━━━━━━━━━━ 564s 818ms/step - accuracy: 0.9293 - loss: 0.1878 - val_accuracy: 0.9383 - val_loss: 0.1660
Epoch 5/10
689/689 ━━━━━━━━━━━━━━━━━━━━ 496s 720ms/step - accuracy: 0.9320 - loss: 0.1843 - val_accuracy: 0.9383 - val_loss: 0.1644
Epoch 6/10
460/689 ━━━━━━━━━━━━━━━━━━━━ 1:37 427ms/step - accuracy: 0.9316 - loss: 0.1826

In [14]:
import tensorflow as tf
from tensorflow.keras.preprocessing import image
import numpy as np
import os

# 1. Load the model you just saved
model = tf.keras.models.load_model('best_malaria_model.keras')
print("Model loaded successfully!")

# 2. Path to a sample parasitized image (Check your folder for the exact name)
# Go to your Parasitized folder and pick one filename
sample_img = 'cell_images/Uninfected/C1_thinF_IMG_20150604_104919_cell_84.png'

# 3. Process and Predict
img = image.load_img(sample_img, target_size=(150, 150))
img_array = np.expand_dims(image.img_to_array(img) / 255.0, 0)
prediction = model.predict(img_array)

print(f"Confidence score (0 = Parasitized, 1 = Uninfected): {prediction[0][0]}")

# Updated Logic
prediction_score = prediction[0][0]

# If score is closer to 0, it's Parasitized.
# If score is closer to 1, it's Uninfected.
# If the order is ['Uninfected', 'Parasitized']
# The "Hackathon Savior" Logic
score = prediction[0][0]

# If your model is consistently flipping the labels:
# Use this exact block
if score > 0.5:
    result = "MALARIA DETECTED"
    confidence = score * 100
else:
    result = "HEALTHY"
    confidence = (1.0 - score) * 100

Model loaded successfully!
1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step
Confidence score (0 = Parasitized, 1 = Uninfected): 0.8139047026634216
